# Fit the DC4 parallel-chunks source catalog

The checked-in YAML contains 71 **full-observation** source models. Its JSON recipe specifies a separate time-weighted response for each variable source or component. This notebook applies a selected GTI to the spacecraft history and matching events, then builds those responses **in memory**. Neither catalog product changes with the GTI.

Default mode reproduces the updated NGC 4151 CPL fit with its existing, already selected three-month mock and background histograms. In `chosen_source` mode, supply full unbinned source and background FITS made from an injection matching this catalog; the available ordinary full DC4 mock FITS has the earlier NGC 4151 injection and is **not** a matching default. A source other than NGC 4151 fits only one overall amplitude, preserving spectral/component ratios. No SED or presentation plots are generated.

The existing NGC 4151 extended-response cube is large (about 13 GiB). Creating a new GTI extended response can take substantial time and disk space.


In [ ]:
from pathlib import Path
from copy import deepcopy
import hashlib
import json
import os
import sys

import astropy.units as u
from astropy.coordinates import SkyCoord
from astropy.io import fits
from astromodels import Cutoff_powerlaw, ExtendedSource, Line, Model, Parameter, PointSource
from astromodels.core.model_parser import ModelParser
from histpy import Histogram, Axis, Axes, HealpixAxis
from threeML import DataList, JointLikelihood
import numpy as np
import pandas as pd
import yaml

from cosipy.background_estimation import FreeNormBinnedBackground
from cosipy.data_io import EmCDSBinnedData
from cosipy.event_selection import GoodTimeInterval
from cosipy.interfaces import ThreeMLPluginInterface
from cosipy.response import (BinnedInstrumentResponse, BinnedThreeMLModelFolding,
    BinnedThreeMLPointSourceResponse, BinnedThreeMLExtendedSourceResponse,
    ExtendedSourceResponse, ZeroPointSourceResponse, ComponentPointSourceResponse,
    make_weighted_point_source_response)
from cosipy.response.FullDetectorResponse import FullDetectorResponse
from cosipy.response.temporal_profile import LightcurveProfile
from cosipy.spacecraftfile import SpacecraftHistory
from cosipy.statistics import PoissonLikelihood
from cosipy.threeml.custom_functions import GalpropHealpixModel, SpecFromDat  # model parser registration
from cosipy.util.megalib import megalib_map_to_galprop_fits


## 1. Set local roots, GTI, and matching data

Set `MODE = "chosen_source"` and `TARGET_SOURCE` to analyze another source. A point source gets its catalog coordinate automatically; for an extended target provide `TARGET_COORD_GALACTIC_DEG = (l, b)`. The FITS paths in that mode must be native, unbinned, time-tagged events from an injection consistent with this YAML. The notebook checks presence and format, while physical injection provenance remains the analyst's responsibility.


In [ ]:
NOTEBOOK_DIRECTORY = Path.cwd().resolve()
if not (NOTEBOOK_DIRECTORY / 'README_source_catalog.md').exists():
    NOTEBOOK_DIRECTORY = NOTEBOOK_DIRECTORY / 'docs/tutorials/spectral_fits/continuum_fit/AGN'
DC4_FILES_ROOT = Path(os.environ.get('DC4_FILES_ROOT', Path.home() / (
    'Library/CloudStorage/OneDrive-ClemsonUniversity/COSI/Radio_Quiet_AGN/DC4_Files')))
SOURCE_LIBRARY = Path(os.environ.get('COSI_SIM_SOURCE_LIBRARY',
    Path.home() / 'Software/cosi-sim/cosi_sim/Source_Library'))
CATALOG_PATH = NOTEBOOK_DIRECTORY / 'source_catalog_DC4_parallel_full.yaml'
RECIPE_PATH = CATALOG_PATH.with_suffix('.response.json')
TEMPLATE_DIRECTORY = DC4_FILES_ROOT / 'DC4_Sources/DC4_Spatial_Templates'
MODE = 'ngc4151_reference'  # or 'chosen_source'
TARGET_SOURCE = 'ngc4151'
TARGET_COORD_GALACTIC_DEG = None  # required for an extended target
MAX_OFFAXIS_DEG = 60.0
EARTH_OCCULTATION = True
EXPOSURE_MULTIPLIER = 1.0

# These two histograms were already selected with the 60-degree NGC 4151 GTI.
REFERENCE_MOCK_HDF5 = DC4_FILES_ROOT / (
    'Mock_Data/Mock_Data_Cut/mock_with_updated_NGC4151_FluxNTH_0p30_time_cut.hdf5')
REFERENCE_BACKGROUND_HDF5 = DC4_FILES_ROOT / (
    'Background/Total_DC4_BG_3months_binned_data_filtered_with_SAAcut_'
    'withSAAbck_NGC4151_60deg_fov_cut.hdf5')
# Set BOTH of these for chosen_source mode. Do not use the ordinary full DC4
# mock FITS here: it contains the earlier NGC 4151 injection.
MATCHING_SOURCE_EVENT_FITS = None
MATCHING_BACKGROUND_EVENT_FITS = None
EXTENDED_CACHE_DIRECTORY = DC4_FILES_ROOT / 'Extended_Responses/SourceCat_cache'
REFERENCE_EXTENDED_RESPONSE = DC4_FILES_ROOT / (
    'Extended_Responses/NGC4151_Cut/'
    'DC4_continuum_NGC4151_60deg_GTI_extended_response_nside8.h5')

if MODE not in {'ngc4151_reference', 'chosen_source'}:
    raise ValueError(f'Unknown MODE: {MODE}')
if MODE == 'ngc4151_reference':
    if TARGET_SOURCE != 'ngc4151' or MAX_OFFAXIS_DEG != 60 or not EARTH_OCCULTATION:
        raise ValueError('The reference HDF5 is valid only for the 60-degree, Earth-occluded NGC 4151 GTI')
else:
    for label, path in [('source', MATCHING_SOURCE_EVENT_FITS),
                        ('background', MATCHING_BACKGROUND_EVENT_FITS)]:
        if path is None or not str(path).endswith(('.fits', '.fits.gz')):
            raise ValueError(f'Chosen-source mode requires matching native unbinned {label} FITS')
        if not Path(path).is_file():
            raise FileNotFoundError(path)
if not np.isfinite(EXPOSURE_MULTIPLIER) or EXPOSURE_MULTIPLIER <= 0:
    raise ValueError('EXPOSURE_MULTIPLIER must be positive')
for path in (CATALOG_PATH, RECIPE_PATH, SOURCE_LIBRARY, DC4_FILES_ROOT):
    if not Path(path).exists():
        raise FileNotFoundError(path)


## 2. Resolve the portable YAML and recipe

The two files stay unchanged on disk. Before parsing, `SOURCE_LIBRARY/` and `DC4_TEMPLATES/` tokens are replaced in memory. Missing spatial templates are recreated from the coupled MEGAlib map. The JSON source list, source/component keys, selection, and catalog filename are checked.


In [ ]:
SPATIAL_TEMPLATE_NSIDE = 8
SPATIAL_TEMPLATE_FORMAT_TAG = "perMeV_v3"
SPATIAL_DEFINITIONS = {'positrons_26al_line': ('DC3', 'Positrons_from_26Al_line'),
 'positrons_26al_cont': ('DC3', 'Positrons_from_26Al_cont'),
 'positrons_44ti_line': ('DC3', 'Positrons_from_44Ti_line'),
 'positrons_44ti_cont': ('DC3', 'Positrons_from_44Ti_cont'),
 'narrow_bulge_511': ('DC3', 'NB_511'),
 'broad_bulge_511': ('DC3', 'BB_511'),
 'vela_snr_511': ('DC3', 'Vela_SNR_511'),
 'al26_ne2001': ('DC3', '26Al_NE2001'),
 'fe60_ne2001': ('DC3', '60Fe_NE2001'),
 'positrons_thin_disk_cont': ('DC4', 'Positrons_Thin_Disk_cont'),
 'positrons_thin_disk_line': ('DC4', 'Positrons_Thin_Disk_line')}

def resolve_source_asset(source_definition, filename):
    path = source_definition.parent / filename
    candidates = [path, Path(f"{path}.gz")]
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    raise FileNotFoundError(
        f"Could not resolve {filename!r} next to {source_definition}"
    )


def parse_megalib_blocks(source_definition):
    # Return one property dictionary per DataChallenge.Source block.
    blocks = []
    by_megalib_name = {}
    for raw_line in source_definition.read_text().splitlines():
        line = raw_line.split("#", 1)[0].strip()
        if not line:
            continue
        tokens = line.split()
        if tokens[:1] == ["DataChallenge.Source"]:
            megalib_name = tokens[1]
            block = {"_name": megalib_name}
            blocks.append(block)
            by_megalib_name[megalib_name] = block
            continue
        if "." not in tokens[0]:
            continue
        megalib_name, prop = tokens[0].split(".", 1)
        if megalib_name in by_megalib_name:
            by_megalib_name[megalib_name][prop] = tokens[1:]
    if not blocks:
        raise ValueError(f"No DataChallenge.Source blocks in {source_definition}")
    return blocks


def source_library_path(value, source_library):
    """Relocate a catalog asset without relying on the original user's home."""
    parts = Path(value).parts
    if "Source_Library" not in parts:
        raise ValueError(f"Not a source-library path: {value}")
    relative = Path(*parts[parts.index("Source_Library") + 1:])
    candidate = (Path(source_library) / relative).resolve()
    if not candidate.is_file():
        raise FileNotFoundError(candidate)
    return str(candidate)


def ensure_spatial_template(source_name, output_path, source_library, overwrite=False):
    """Build only missing templates, preserving the generator's conversion."""
    output_path = Path(output_path)
    if output_path.exists() and not overwrite:
        with fits.open(output_path) as hdul:
            if (hdul['SKYMAP'].header.get('NSIDE') != SPATIAL_TEMPLATE_NSIDE
                    or hdul['SKYMAP'].header.get('MAPVERS') != SPATIAL_TEMPLATE_FORMAT_TAG):
                raise ValueError(f"Incompatible cached spatial template: {output_path}")
        return output_path, 'loaded'
    challenge, stem = SPATIAL_DEFINITIONS[source_name]
    matches = sorted((Path(source_library) / challenge / 'sources').rglob(f'{stem}.source'))
    if len(matches) != 1:
        raise ValueError(f"Expected one {stem}.source, found {matches}")
    blocks = parse_megalib_blocks(matches[0])
    if len(blocks) != 1 or len(blocks[0].get('Beam', [])) < 2:
        raise ValueError(f"Expected one coupled map in {matches[0]}")
    map_path = resolve_source_asset(matches[0], blocks[0]['Beam'][1])
    megalib_map_to_galprop_fits(
        map_path, output_path, nside=SPATIAL_TEMPLATE_NSIDE,
        oversample_nside=64, energy_bounds=(100., 10000.), overwrite=overwrite,
    )
    return output_path, 'created'



def resolve_token(value):
    value = str(value)
    prefixes = {
        'SOURCE_LIBRARY/': SOURCE_LIBRARY,
        'DC4_FILES/': DC4_FILES_ROOT,
        'DC4_TEMPLATES/': TEMPLATE_DIRECTORY,
    }
    for prefix, root in prefixes.items():
        if value.startswith(prefix):
            relative = Path(value[len(prefix):])
            if '..' in relative.parts or relative.is_absolute():
                raise ValueError(f'Unsafe asset path: {value}')
            return root / relative
    raise ValueError(f'Unsupported catalog asset token: {value}')


def load_portable_catalog():
    contents = yaml.safe_load(CATALOG_PATH.read_text())
    recipe = json.loads(RECIPE_PATH.read_text())
    if (recipe.get('schema') != 'dc4-component-lightcurve-response-v2'
            or recipe.get('selection') != 'full' or recipe.get('time_basis') != 'elapsed'
            or recipe.get('catalog') != CATALOG_PATH.name):
        raise ValueError('The YAML/JSON pair is mismatched or has an unsupported schema')
    def relocate(node, source_name):
        if isinstance(node, dict):
            for key, value in node.items():
                if key == '_fitsfile' and isinstance(value, str):
                    output = resolve_token(value)
                    ensure_spatial_template(source_name, output, SOURCE_LIBRARY)
                    node[key] = str(output.resolve())
                elif isinstance(value, str) and value.startswith('SOURCE_LIBRARY/'):
                    path = resolve_token(value)
                    if not path.is_file():
                        raise FileNotFoundError(path)
                    node[key] = str(path.resolve())
                else:
                    relocate(value, source_name)
        elif isinstance(node, list):
            for value in node:
                relocate(value, source_name)
    for key, definition in contents.items():
        relocate(definition, key.split(' (')[0])
    model = ModelParser(model_dict=contents).get_model()
    keys = [(r['source'], r['component']) for r in recipe['components']]
    if (len(model.sources) != 71 or len(keys) != 74 or len(set(keys)) != 74
            or set(r['source'] for r in recipe['components']) != set(model.sources)
            or any(r['selection'] != 'full' for r in recipe['components'])):
        raise ValueError('Catalog and recipe source/component membership differs')
    if model.free_parameters:
        raise ValueError('The injected source catalog must be frozen')
    for row in recipe['components']:
        if row['lightcurve']:
            path = resolve_token(row['lightcurve'])
            if not path.is_file():
                raise FileNotFoundError(path)
            row['lightcurve'] = str(path.resolve())
    return model, recipe

model, recipe = load_portable_catalog()
ORIENTATION_PATH = resolve_token(recipe['orientation'])
RESPONSE_PATH = resolve_token(recipe['detector_response'])
for path in (ORIENTATION_PATH, RESPONSE_PATH):
    if not path.is_file():
        raise FileNotFoundError(path)
print(f'Loaded {len(model.point_sources)} point and {len(model.extended_sources)} extended sources')


## 3. Select data and spacecraft history

Every response uses the same GTI-cut pointing. Variable-source responses then weight each interval by its own light curve divided by that component's **full-observation** factor. Repeating the exposure scales source and background counts and both response types together.


In [ ]:
def load_selected_histogram(path, gti, binning, already_selected=True):
    """Load explicitly preselected HDF5 or select native FITS events before binning.

    Histogram axes do not encode their GTI. For an already-selected histogram,
    its time-selection provenance is the caller's responsibility. An unselected
    projected histogram is rejected because event times cannot be recovered.
    Native event selection uses half-open [start, stop) GTI intervals.
    """
    import astropy.units as u
    from astropy.coordinates import SkyCoord
    from histpy import Histogram, Axis, Axes, HealpixAxis

    config = {'energy_bins': binning['energy_bins_keV'],
              'phi_pix_size': binning['phi_pix_size_deg'],
              'nside': binning['nside'], 'scheme': binning['scheme']}
    axes = Axes([
        Axis(config['energy_bins'], unit=u.keV, label='Em'),
        Axis(np.linspace(0., 180., int(180. / config['phi_pix_size']) + 1),
             unit=u.deg, label='Phi'),
        HealpixAxis(nside=config['nside'], scheme=config['scheme'],
                    coordsys='galactic', label='PsiChi'),
    ])
    path = Path(path)
    is_fits = path.name.endswith(('.fits', '.fits.gz'))
    if not is_fits:
        if not already_selected:
            raise ValueError('Use native event FITS to apply a new GTI, not a projected histogram')
        histogram = Histogram.open(path).project('Em', 'Phi', 'PsiChi')
        if histogram.axes != axes:
            raise ValueError(f'Histogram binning differs from agn.yaml: {path}')
        return histogram

    starts = np.asarray(gti.tstart_list.unix, dtype=float)
    stops = np.asarray(gti.tstop_list.unix, dtype=float)
    if (starts.size == 0 or np.any(stops <= starts)
            or np.any(starts[1:] < stops[:-1])):
        raise ValueError('GTI intervals must be positive, sorted and nonoverlapping')
    histogram = Histogram(axes, sparse=True)
    with fits.open(path, memmap=False) as hdul:
        events = hdul[1].data
        for offset in range(0, len(events), 250_000):
            block = events[offset:offset + 250_000]
            times = np.asarray(block['TimeTags'], dtype=float)
            if already_selected:
                selected = np.ones(times.size, dtype=bool)
            else:
                index = np.searchsorted(starts, times, side='right') - 1
                selected = (index >= 0) & (times < stops[np.maximum(index, 0)])
            direction = SkyCoord(
                l=np.asarray(block['Chi galactic'][selected]) * u.deg,
                b=np.asarray(block['Psi galactic'][selected]) * u.deg,
                frame='galactic',
            )
            histogram.fill(np.asarray(block['Energies'][selected]) * u.keV,
                           np.asarray(block['Phi'][selected]) * u.rad, direction)
    return histogram

if TARGET_SOURCE not in model.sources:
    raise ValueError(f'Unknown catalog source: {TARGET_SOURCE}')
target = model.sources[TARGET_SOURCE]
if isinstance(target, PointSource):
    coord = target.position.sky_coord
elif TARGET_COORD_GALACTIC_DEG is not None:
    coord = SkyCoord(l=TARGET_COORD_GALACTIC_DEG[0]*u.deg,
                     b=TARGET_COORD_GALACTIC_DEG[1]*u.deg, frame='galactic')
else:
    raise ValueError('Extended target requires TARGET_COORD_GALACTIC_DEG = (l, b)')
full_history = SpacecraftHistory.open(ORIENTATION_PATH)
gti = GoodTimeInterval.from_pointing_cut(
    coord, full_history, MAX_OFFAXIS_DEG*u.deg, earth_occ=EARTH_OCCULTATION)
cut_history = full_history.apply_gti(gti)
if cut_history.nintervals == 0:
    raise ValueError('The chosen GTI has no spacecraft intervals')
scaled_history = SpacecraftHistory(
    cut_history.obstime, cut_history.attitude, cut_history.location,
    livetime=cut_history.livetime * EXPOSURE_MULTIPLIER)
scaled_history.cache_earth_occ = cut_history.cache_earth_occ
if MODE == 'ngc4151_reference':
    data_path, background_path = REFERENCE_MOCK_HDF5, REFERENCE_BACKGROUND_HDF5
    already_selected = True
else:
    data_path, background_path = Path(MATCHING_SOURCE_EVENT_FITS), Path(MATCHING_BACKGROUND_EVENT_FITS)
    already_selected = False
for path in (data_path, background_path):
    if not path.is_file():
        raise FileNotFoundError(path)
data_hist = load_selected_histogram(data_path, gti, recipe['binning'], already_selected)
total_bkg = load_selected_histogram(background_path, gti, recipe['binning'], already_selected)
if data_hist.axes != total_bkg.axes:
    raise ValueError('Source data and background data use different axes')
data_hist *= EXPOSURE_MULTIPLIER
total_bkg *= EXPOSURE_MULTIPLIER
total_bkg += sys.float_info.min
background_rate_initial = float(total_bkg.to_dense(copy=False).contents.sum()) / float(
    scaled_history.cumulative_livetime().to_value(u.s))
print('GTI intervals:',cut_history.nintervals,'observed counts:',float(data_hist.to_dense(copy=False).contents.sum()))


## 4. Build ordinary and source-specific responses

Steady point sources share one ordinary history. Each variable source (and each differently varying Crab component) gets its own in-memory weighted history. The extended response cache is keyed by the GTI interval edges, orientation and detector-response files, binning, Earth-occultation setting. The precomputed NGC 4151 reference cube is used only for the documented reference GTI.


In [ ]:
def component_name(source, label):
    if label in source.components:
        return label
    # v1 recipes store MEGAlib labels rather than astromodels component names.
    crab_names = {'Crab_Nebula': 'nebula', 'Crab_P1': 'peak1',
                  'Crab_Brg': 'bridge', 'Crab_P2': 'peak2'}
    if source.name == 'crab' and crab_names.get(label) in source.components:
        return crab_names[label]
    raise ValueError(f"Unknown temporal component {source.name}/{label}")


def make_temporal_responses(model, recipe, base_history, gti, data,
                            instrument_response, detector_response,
                            exposure_multiplier=1., full_history=None, earth_occ=True):
    """Build per-source/component responses from the unscaled GTI history.

    Validate temporal factors over the CATALOG selection, which can differ
    from the analysis GTI. Full catalogs require the original full_history:
    response weights remain L(t)/mean_full while exposure is GTI restricted.
    Exposure repetitions multiply response livetimes, not fluxes.
    Responses are computed lazily by cosipy and reused throughout fitting.
    """
    if not np.isfinite(exposure_multiplier) or exposure_multiplier <= 0:
        raise ValueError("Exposure multiplier must be finite and positive")
    starts, stops = np.asarray(gti.tstart_list.unix), np.asarray(gti.tstop_list.unix)
    duration = float(np.sum(stops-starts))
    if duration <= 0 or np.any(stops <= starts):
        raise ValueError("The analysis GTI must have positive duration")
    catalog_selection = recipe.get('selection', 'NGC4151_GTI')
    if catalog_selection == 'full':
        if full_history is None:
            raise ValueError("Full-observation catalogs require the original full_history")
        reference_starts = np.array([full_history.obstime[0].unix])
        reference_stops = np.array([full_history.obstime[-1].unix])
        if np.any(starts < reference_starts[0]-1e-5) or np.any(stops > reference_stops[0]+1e-5):
            raise ValueError("The analysis GTI extends beyond the catalog observation")
    elif catalog_selection == 'NGC4151_GTI':
        reference_starts, reference_stops = starts, stops
    else:
        raise ValueError(f"Unsupported catalog selection: {catalog_selection}")
    reference_duration = float(np.sum(reference_stops-reference_starts))
    if not np.isfinite(reference_duration) or reference_duration <= 0:
        raise ValueError("Invalid catalog reference duration")
    history_starts, history_stops = base_history.intervals_tstart.unix, base_history.intervals_tstop.unix
    rows_by_source = {}
    for row in recipe['components']:
        rows_by_source.setdefault(row['source'], []).append(row)
    overrides, diagnostics = {}, []
    for key, rows in rows_by_source.items():
        source = model.sources[key]
        for row in rows:
            if not np.isclose(row['selected seconds'], reference_duration, rtol=0., atol=1e-4):
                raise ValueError(f"Reference duration differs from the catalog: {key}")
        if not any(row['lightcurve'] for row in rows):
            continue
        if not isinstance(source, PointSource):
            raise NotImplementedError(f"A temporal extended response is required for {key}")
        if any(r['component'] == 'all' for r in rows) and len(rows) != 1:
            raise ValueError(f"Invalid component grouping for {key}")
        component_responses = {}
        for row in rows:
            applied = float(row['applied factor'])
            if not np.isfinite(applied) or applied <= 0:
                raise ValueError(f"Invalid applied factor for {key}")
            if row['lightcurve']:
                profile = LightcurveProfile.from_dat(
                    row['lightcurve'], expected_sha256=row['lightcurve sha256'],
                    repeating=row['repeating'], mode=row['normalization mode'],
                    bins=int(row['simulation bins']),
                )
                mean = float(profile.integral(reference_starts, reference_stops).sum()/reference_duration)
                gti_mean = float(profile.integral(starts, stops).sum()/duration)
                weights = profile.response_weights(history_starts, history_stops, applied)
                del profile
            else:
                mean, gti_mean, weights = 1., 1., np.ones(base_history.nintervals)
            if not np.isclose(mean, row['time factor'], rtol=1e-8, atol=1e-14):
                raise ValueError(f"Temporal factor differs from catalog for {key}/{row['component']}")
            if not np.isclose(applied, max(mean, 1e-30), rtol=1e-8, atol=0.):
                raise ValueError(f"Applied factor differs from catalog for {key}")
            weighted_live = base_history.livetime * weights * exposure_multiplier
            response = make_weighted_point_source_response(
                base_history, weights, data, instrument_response, detector_response,
                exposure_multiplier=exposure_multiplier, earth_occ=earth_occ,
            )
            diagnostics.append({'source': key, 'component': row['component'],
                                'normalization mode': row['normalization mode'],
                                'simulation bins': row['simulation bins'],
                                'catalog selection': catalog_selection,
                                'catalog time factor': mean,
                                'analysis GTI time factor': gti_mean,
                                'response denominator': applied,
                                'weighted response seconds': float(weighted_live.to_value(u.s).sum())})
            if row['component'] == 'all':
                overrides[key] = response
            else:
                name = component_name(source, row['component'])
                if name in component_responses:
                    raise ValueError(f"Duplicate mapped component for {key}/{name}")
                component_responses[name] = response
        if component_responses:
            if set(component_responses) != set(source.components):
                raise ValueError(f"Recipe does not cover all components of {key}")
            overrides[key] = ComponentPointSourceResponse(component_responses)
    return overrides, diagnostics

def extended_cache_key():
    payload = {
        'gti_starts': np.asarray(gti.tstart_list.unix).tolist(),
        'gti_stops': np.asarray(gti.tstop_list.unix).tolist(),
        'orientation': str(ORIENTATION_PATH.resolve()),
        'orientation_stat': (ORIENTATION_PATH.stat().st_size, ORIENTATION_PATH.stat().st_mtime_ns),
        'detector_response': str(RESPONSE_PATH.resolve()),
        'detector_response_stat': (RESPONSE_PATH.stat().st_size, RESPONSE_PATH.stat().st_mtime_ns),
        'binning': recipe['binning'],
        'earth_occultation': EARTH_OCCULTATION,
        'nside_scatt_map': 2*data.axes['PsiChi'].nside,
        'nside_image': 8,
    }
    return hashlib.sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest()[:20]

# The comparison reference folded all sources unpolarized. Keep that exact
# approximation; polarization parameters remain fixed in the catalog.
for source in model.sources.values():
    for component in source.components.values():
        degree = getattr(component.polarization, 'degree', None)
        if degree is not None:
            degree.value = 0.0

data = EmCDSBinnedData(data_hist)
dr = FullDetectorResponse.open(str(RESPONSE_PATH))
instrument_response = BinnedInstrumentResponse(dr, data)
psr = BinnedThreeMLPointSourceResponse(
    data=data, instrument_response=instrument_response, sc_history=scaled_history,
    energy_axis=dr.axes['Ei'],
    polarization_axis=dr.axes['Pol'] if 'Pol' in dr.axes.labels else None,
    nside=2*data.axes['PsiChi'].nside, earth_occ=EARTH_OCCULTATION)
cache_path = EXTENDED_CACHE_DIRECTORY / f'dc4_extended_{extended_cache_key()}.h5'
if MODE == 'ngc4151_reference' and REFERENCE_EXTENDED_RESPONSE.is_file():
    # Provenance of this legacy cache is the exact reference GTI above.
    cache_path = REFERENCE_EXTENDED_RESPONSE
if cache_path.is_file():
    extended_response = ExtendedSourceResponse.open(cache_path)
else:
    extended_response = dr.get_extended_source_response(
        cut_history, coordsys='galactic', nside_image=8,
        nside_scatt_map=2*data.axes['PsiChi'].nside,
        earth_occ=EARTH_OCCULTATION)
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    extended_response.write(cache_path, overwrite=False)
if (extended_response.axes['NuLambda'].nside != 8
        or extended_response.axes[2:] != data.axes
        or extended_response.axes['Ei'] != dr.axes['Ei']):
    raise ValueError(f'Extended response axes differ from data: {cache_path}')
# The cached cube always represents one three-month exposure. Do not persist
# the in-memory scaled copy back into this cache.
extended_response *= EXPOSURE_MULTIPLIER
esr = BinnedThreeMLExtendedSourceResponse(data=data, precomputed_psr=extended_response)
source_specific, temporal_diagnostics = make_temporal_responses(
    model, recipe, cut_history, gti, data, instrument_response, dr,
    exposure_multiplier=EXPOSURE_MULTIPLIER, full_history=full_history,
    earth_occ=EARTH_OCCULTATION)
response = BinnedThreeMLModelFolding(
    data=data, point_source_response=psr, extended_source_response=esr,
    source_specific_point_source_responses=source_specific)
print('Source-specific responses:',len(source_specific),'extended cache:',cache_path)


## 5. Free only the requested spectral parameters

For the NGC 4151 reference, replace its injected CPL+PL spectrum in memory with a CPL and fit `K`, `index`, and `xc`. For another source, keep its injected shape and free one amplitude. If that source has several spectral normalizations, fixed linear links to the first amplitude preserve all their original ratios.


In [ ]:
def source_amplitudes(source):
    spatial = getattr(source, 'spatial_shape', None)
    if isinstance(spatial, GalpropHealpixModel):
        return [spatial.K]
    values = []
    for component in source.components.values():
        pars = list(component.shape.parameters.values())
        amps = [p for p in pars if p.is_normalization]
        if not amps:
            amps = [p for p in pars if p.name.split('_', 1)[0] in {'F','K','k'}]
        values.extend(amps)
    if not values:
        raise ValueError(f'No multiplicative amplitude in {source.name}')
    return values

if MODE == 'ngc4151_reference':
    original = model.ngc4151
    shape = original.spectrum.main.composite
    cpl = Cutoff_powerlaw(K=shape.K_1.value, piv=shape.piv_1.value,
                          index=shape.index_1.value, xc=shape.xc_1.value)
    replacement = PointSource('ngc4151', l=original.position.l.value,
                              b=original.position.b.value, spectral_shape=cpl)
    for p in replacement.parameters.values():
        p.fix = True
    model.remove_source('ngc4151')
    model.add_source(replacement)
    shape = model.ngc4151.spectrum.main.Cutoff_powerlaw
    shape.K.min_value, shape.K.max_value = 1e-8, 1e-2
    shape.K.delta, shape.K.fix = 0.1*shape.K.value, False
    shape.index.min_value, shape.index.max_value = -10., 10.
    shape.index.delta, shape.index.fix = 0.1, False
    shape.xc.min_value, shape.xc.max_value = 100., 10000.
    shape.xc.delta, shape.xc.fix = 0.1*shape.xc.value, False
    shape.piv.fix = True
    source_fit_parameters = [shape.K, shape.index, shape.xc]
else:
    amplitudes = source_amplitudes(model.sources[TARGET_SOURCE])
    master = amplitudes[0]
    initial = float(master.value)
    if initial <= 0:
        raise ValueError('Cannot fit a multiplicative scale from a zero source amplitude')
    master.min_value = min(initial/1000., float(master.min_value or initial/1000.))
    master.max_value = max(initial*100., float(master.max_value or initial*100.))
    master.delta = max(initial*.1, 1e-15)
    master.fix = False
    for amplitude in amplitudes[1:]:
        ratio = float(amplitude.value) / initial
        law = Line()
        law.a.value, law.a.fix = 0., True
        law.b.value, law.b.fix = ratio, True
        model.link(amplitude, master, link_function=law)
    source_fit_parameters = [master]
if set(model.free_parameters) != {p.path for p in source_fit_parameters}:
    raise ValueError(f'Unexpected free source parameters: {list(model.free_parameters)}')
print('Free source parameters:',[p.path for p in source_fit_parameters])


## 6. Fit and report valid uncertainties and counts

The optimizer fits the source parameter(s) and one instrumental-background rate. Uncertainties are taken from the fitted covariance. If covariance is unavailable or invalid, the notebook reports that explicitly; it never prints a zero as a stand-in for uncertainty. The final table projects both observed and forward-folded counts into the native ten measured-energy bins.


In [ ]:
background = FreeNormBinnedBackground(
    {'total_bkg': total_bkg}, sc_history=scaled_history, copy=True)
likelihood = PoissonLikelihood(data, response, background)
plugin = ThreeMLPluginInterface('cosi', likelihood, response, background)
plugin.bkg_parameter['total_bkg'] = Parameter(
    'total_bkg', background_rate_initial,
    min_value=0., max_value=max(50., 10*background_rate_initial),
    delta=max(.05*background_rate_initial, 1e-6), unit=u.Hz)
joint = JointLikelihood(model, DataList(plugin), verbose=False)
from threeML.exceptions.custom_exceptions import BadCovariance
from threeML.minimizer.minimization import CannotComputeCovariance
covariance_valid = True
try:
    _ = joint.fit(compute_covariance=True)
except (BadCovariance, CannotComputeCovariance) as exc:
    # A rejected covariance cannot supply valid error bars. Refit without it,
    # then request profile-likelihood intervals below. An optimizer failure
    # still propagates; it is never presented as a successful fit.
    print(f'Covariance unavailable ({type(exc).__name__}); using profile intervals')
    covariance_valid = False
    _ = joint.fit(compute_covariance=False)
joint.restore_best_fit()
results = joint.results
if covariance_valid:
    matrix = np.asarray(joint._minimizer.covariance_matrix, dtype=float)
    covariance_valid = (matrix.ndim == 2 and np.all(np.isfinite(matrix))
                        and np.all(np.linalg.eigvalsh(matrix) > 0))
    if not covariance_valid:
        print('Covariance is singular or missing; using profile intervals')
profile_errors = None
if not covariance_valid:
    try:
        profile_errors = joint.get_errors(quiet=True)
    except Exception as exc:
        print(f'Profile intervals unavailable: {type(exc).__name__}: {exc}')

parameter_rows = []
for parameter in [*source_fit_parameters, plugin.bkg_parameter['total_bkg']]:
    path = parameter.path
    value = float(parameter.value)
    uncertainty = None
    status = 'unavailable: invalid or non-positive covariance interval'
    try:
        if covariance_valid:
            variate = results.get_variates(path)
            lo, hi = variate.equal_tail_interval(cl=.68)
            lo, hi = float(lo), float(hi)
            if np.isfinite(lo) and np.isfinite(hi) and hi > lo and lo <= value <= hi:
                uncertainty = (value-lo, hi-value)
                status = 'valid covariance 68% interval'
        elif profile_errors is not None:
            matches = [key for key in profile_errors.index
                       if key == path or key.endswith('.' + path)]
            if len(matches) != 1:
                raise KeyError(f'No unique profile interval for {path}')
            lower = float(profile_errors.loc[matches[0], 'negative_error'])
            upper = float(profile_errors.loc[matches[0], 'positive_error'])
            if np.isfinite(lower) and np.isfinite(upper) and lower < 0 < upper:
                uncertainty = (-lower, upper)
                status = 'valid profile interval'
    except Exception as exc:
        status = f'unavailable: {type(exc).__name__}'
    parameter_rows.append({'parameter': path, 'value': value,
                           'minus 68%': None if uncertainty is None else uncertainty[0],
                           'plus 68%': None if uncertainty is None else uncertainty[1],
                           'status': status})
parameter_table = pd.DataFrame(parameter_rows)
print(parameter_table.to_string(index=False))

observed = np.asarray(data_hist.project('Em').to_dense(copy=False).contents, dtype=float)
source_counts = np.asarray(response.expectation().project('Em').to_dense(copy=False).contents, dtype=float)
background_counts = np.asarray(background.expectation(copy=True).project('Em').to_dense(copy=False).contents, dtype=float)
energy_edges = data_hist.axes['Em'].edges.to_value(u.keV)
counts_table = pd.DataFrame({
    'E low (keV)': energy_edges[:-1], 'E high (keV)': energy_edges[1:],
    'observed': observed, 'source model': source_counts,
    'background model': background_counts,
    'total model': source_counts+background_counts,
    'residual': observed-source_counts-background_counts,
})
print(f'Log likelihood: {float(plugin.get_log_like()):.6g}')
print(counts_table.to_string(index=False))
